# 01. Tensor와 GPU 연산

RunPod Jupyter에서 각 셀을 순서대로 실행한다. 설치는 이미 완료됐다고 가정한다.

이번 실습: **텐서의 형태·위치·행렬곱, CPU/GPU 실행시간, VRAM 사용량**을 직접 확인한다.

In [ ]:
import time
import torch

assert torch.cuda.is_available(), "GPU가 활성화된 Jupyter 커널을 선택하세요."
device = torch.device("cuda")
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))

## 텐서의 실제 값과 구조

`shape`는 차원별 원소 개수, `stride`는 각 차원의 인덱스가 1 증가할 때 이동하는 원소 간격이다. 텐서의 위치는 `device`로 확인한다.

In [ ]:
x = torch.arange(12, dtype=torch.float32, device=device).reshape(3, 4)
print(x)
print("shape:", x.shape, "stride:", x.stride(), "dtype:", x.dtype, "device:", x.device)
print("transpose shape/stride:", x.T.shape, x.T.stride())
print("x @ x.T:\n", x @ x.T)

## 같은 행렬곱을 CPU와 GPU에서 계산

측정 구간에는 행렬을 CPU에서 GPU로 복사하는 시간이 포함되지 않는다. CUDA는 비동기 실행이므로 시간을 재기 전에 `torch.cuda.synchronize()`를 사용한다.

In [ ]:
n = 1024         # 2048, 4096으로 바꿔본다. 메모리 부족 시 줄인다.
repeats = 20
torch.manual_seed(42)

a_cpu = torch.randn(n, n)
b_cpu = torch.randn(n, n)
a_gpu = a_cpu.to(device)
b_gpu = b_cpu.to(device)

def benchmark(a, b, repeats):
    # 첫 실행의 초기화 비용을 줄이기 위해 워밍업
    for _ in range(3):
        _ = a @ b
    if a.is_cuda:
        torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(repeats):
        out = a @ b
    if a.is_cuda:
        torch.cuda.synchronize()
    return (time.perf_counter() - start) / repeats, out

cpu_sec, cpu_out = benchmark(a_cpu, b_cpu, repeats)
gpu_sec, gpu_out = benchmark(a_gpu, b_gpu, repeats)
print(f"CPU: {cpu_sec * 1000:.3f} ms/회")
print(f"GPU: {gpu_sec * 1000:.3f} ms/회")
print("최대 절대 오차:", (cpu_out - gpu_out.cpu()).abs().max().item())

## VRAM 확인

`allocated`는 살아 있는 PyTorch 텐서가 차지한 메모리이고 `reserved`는 PyTorch 메모리 관리자가 예약한 양이다. `nvidia-smi`와 수치가 다를 수 있다.

In [ ]:
print("allocated MiB:", torch.cuda.memory_allocated() / 2**20)
print("reserved MiB:", torch.cuda.memory_reserved() / 2**20)
print("peak allocated MiB:", torch.cuda.max_memory_allocated() / 2**20)

**직접 바꿔볼 것:** `n`을 1024, 2048, 4096으로 변경하고 실행시간과 VRAM을 비교한다. `torch.randn(n, n, dtype=torch.float16)`으로 자료형을 바꾼 경우도 비교한다. CPU와 GPU가 모든 크기에서 같은 속도 관계를 보이는지 관찰한다.